# Transformer 完全从零实现

本 notebook 从零实现 Transformer 模型，**完全不依赖 d2l 库**，只使用 PyTorch 和标准库。

In [ ]:
import math
import time
import collections
import requests
import zipfile
import os
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. 设备选择

In [ ]:
def try_gpu():
    """返回 GPU（如果可用），否则返回 CPU"""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device = try_gpu()
print(f"Using device: {device}")

## 2. 词表类

In [ ]:
class Vocab:
    """词表类：支持 token 到索引的双向映射"""
    
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        """初始化词表
        
        Args:
            tokens: token 列表的列表
            min_freq: 最小词频，低于此频率的 token 会被标记为 <unk>
            reserved_tokens: 保留的特殊 token
        """
        if reserved_tokens is None:
            reserved_tokens = []
        
        # 特殊标记：按顺序添加
        self.idx_to_token = ['<unk>'] + reserved_tokens + ['<bos>', '<eos>', '<pad>']
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        
        # 统计词频
        counter = collections.Counter()
        if tokens is not None:
            for token_list in tokens:
                counter.update(token_list)
        
        # 按频率排序并添加到词表
        self._freqs = counter
        self.sorted_tokens = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        
        for token, freq in self.sorted_tokens:
            if freq >= min_freq and token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1
    
    def __len__(self):
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens):
        """将 token(s) 转换为索引"""
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    
    def to_tokens(self, indices):
        """将索引转换为 token(s)"""
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]
    
    @property
    def unk(self):
        """未知 token 的索引"""
        return 0
    
    @property
    def bos(self):
        """开始 token 的索引"""
        return self.token_to_idx['<bos>']
    
    @property
    def eos(self):
        """结束 token 的索引"""
        return self.token_to_idx['<eos>']
    
    @property
    def pad(self):
        """填充 token 的索引"""
        return self.token_to_idx['<pad>']

## 3. 数据预处理和加载

In [ ]:
def preprocess_nmt(text):
    """预处理英法翻译文本
    
    - 转小写
    - 添加空格
    - 标准化标点符号
    """
    def no_space(char, prev_char):
        return char in set(',.!?') and prev_char != ' '
    
    # 替换非换行空格，转小写
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    # 在标点前插入空格
    out = []
    for i, char in enumerate(text):
        if i > 0 and no_space(char, text[i - 1]):
            out.append(' ' + char)
        else:
            out.append(char)
    return ''.join(out)


def tokenize_nmt(text, max_examples=None):
    """分词并返回源语言和目标语言列表"""
    source, target = [], []
    for i, line in enumerate(text.split('\n')):
        if max_examples and i > max_examples:
            break
        parts = line.split('\t')
        if len(parts) == 2:
            source.append(parts[0].split(' '))
            target.append(parts[1].split(' '))
    return source, target


def truncate_pad(line, num_steps, padding_token):
    """截断或填充序列到固定长度"""
    if len(line) > num_steps:
        return line[:num_steps]
    return line + [padding_token] * (num_steps - len(line))


def build_array_nmt(lines, vocab, num_steps):
    """将文本序列转换为张量数组"""
    lines = [vocab[l] for l in lines]
    lines = [l + [vocab.eos] for l in lines]
    array = torch.tensor([truncate_pad(l, num_steps, vocab.pad) for l in lines])
    valid_len = (array != vocab.pad).sum(dim=1)
    return array, valid_len


def load_array(data_arrays, batch_size, is_train=True):
    """PyTorch 数据迭代器"""
    dataset = TensorDataset(*data_arrays)
    return DataLoader(dataset, batch_size, shuffle=is_train)


def load_data_nmt(batch_size, num_steps, num_examples=600):
    """加载英法翻译数据集（按照 d2l 的实现方式）
    
    Args:
        batch_size: 批大小
        num_steps: 序列最大长度
        num_examples: 使用的样本数量
    
    Returns:
        train_iter: 数据迭代器
        src_vocab: 源语言词表
        tgt_vocab: 目标语言词表
    """
    # 数据 URL
    url = 'http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip'
    
    # 数据目录（按照 d2l 的方式）
    data_dir = os.path.join(os.getcwd(), 'data', 'fra-eng')
    os.makedirs(data_dir, exist_ok=True)
    zip_path = os.path.join(os.path.dirname(data_dir), 'fra-eng.zip')
    
    # 下载数据
    if not os.path.exists(zip_path):
        print(f"Downloading {url}...")
        response = requests.get(url, timeout=30)
        os.makedirs(os.path.dirname(zip_path), exist_ok=True)
        with open(zip_path, 'wb') as f:
            f.write(response.content)
        print("Download complete.")
    
    # 解压数据
    if not os.path.exists(os.path.join(data_dir, 'fra.txt')):
        print("Extracting data...")
        with zipfile.ZipFile(zip_path, 'r') as fp:
            fp.extractall(os.path.dirname(data_dir))
        print("Extraction complete.")
    
    # 读取数据
    with open(os.path.join(data_dir, 'fra.txt'), 'r', encoding='utf-8') as f:
        text = f.read()
    
    # 预处理
    text = preprocess_nmt(text)
    source, target = tokenize_nmt(text, num_examples)
    
    # 构建词表
    src_vocab = Vocab(source, min_freq=2, reserved_tokens=['<pad>'])
    tgt_vocab = Vocab(target, min_freq=2, reserved_tokens=['<pad>'])
    
    print(f"源语言词表大小: {len(src_vocab)}, 目标语言词表大小: {len(tgt_vocab)}")
    
    # 构建数据数组
    src_array, src_valid_len = build_array_nmt(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array_nmt(target, tgt_vocab, num_steps)
    
    # 创建数据迭代器
    data_arrays = (src_array, src_valid_len, tgt_array, tgt_valid_len)
    train_iter = load_array(data_arrays, batch_size, is_train=True)
    
    return train_iter, src_vocab, tgt_vocab

## 4. 训练辅助类

In [ ]:
class Timer:
    """记录多次运行时间"""
    
    def __init__(self):
        self.times = []
        self.start()
    
    def start(self):
        """启动计时器"""
        self.tik = time.time()
    
    def stop(self):
        """停止计时器并返回时间"""
        self.times.append(time.time() - self.tik)
        return self.times[-1]


class Accumulator:
    """在 n 个变量上累加"""
    
    def __init__(self, n):
        self.data = [0.0] * n
    
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    
    def reset(self):
        self.data = [0.0] * len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


class Animator:
    """使用 matplotlib 绘制训练进度"""
    
    def __init__(self, xlabel=None, ylabel=None, xlim=None, ylim=None,
                 xscale='linear', yscale='linear',
                 legend=None, figsize=(5, 3)):
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.xlabel = xlabel
        self.ylabel = ylabel
        self.xlim = xlim
        self.ylim = ylim
        self.xscale = xscale
        self.yscale = yscale
        self.legend = legend
        self.x_data = []
        self.y_data = [] if legend is None else [[] for _ in legend]
    
    def add(self, x, y):
        """添加数据点"""
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        
        self.x_data.extend(x)
        if self.legend is None:
            self.y_data.extend(y)
        else:
            for i, yi in enumerate(y):
                self.y_data[i].extend(yi)
        
        self.ax.cla()
        if self.xlabel:
            self.ax.set_xlabel(self.xlabel)
        if self.ylabel:
            self.ax.set_ylabel(self.ylabel)
        if self.xlim:
            self.ax.set_xlim(self.xlim)
        if self.ylim:
            self.ax.set_ylim(self.ylim)
        self.ax.set_xscale(self.xscale)
        self.ax.set_yscale(self.yscale)
        
        if self.legend is None:
            self.ax.plot(self.x_data, self.y_data)
        else:
            for i, (label, ys) in enumerate(zip(self.legend, self.y_data)):
                self.ax.plot(self.x_data[:len(ys)], ys, label=label)
            self.ax.legend()
        
        self.fig.canvas.draw()
        plt.pause(0.01)
        
    def show(self):
        plt.show()

## 5. 训练相关函数

In [ ]:
def sequence_mask(X, valid_len, value=0):
    """序列掩码函数"""
    maxlen = X.size(1)
    mask = torch.arange(maxlen, dtype=torch.float32, device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X


def masked_softmax_ce_loss(pred, label, valid_len):
    """带掩码的交叉熵损失
    
    Args:
        pred: (batch_size, num_steps, vocab_size)
        label: (batch_size, num_steps)
        valid_len: (batch_size,) 有效长度
    
    Returns:
        (batch_size,) 每个样本的平均损失
    """
    weights = torch.ones_like(label)
    weights = sequence_mask(weights, valid_len, value=0)
    
    # pred: (B, T, V) -> (B, V, T) for CrossEntropyLoss
    pred = pred.permute(0, 2, 1)
    
    # reduction='none' 意味着不进行任何归约
    unweighted_loss = F.cross_entropy(pred, label, reduction='none')
    weighted_loss = (unweighted_loss * weights).mean(dim=1)
    return weighted_loss


def grad_clipping(net, theta=1.0):
    """梯度裁剪
    
    将所有梯度的范数裁剪到 theta
    """
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.grad is not None]
    else:
        params = list(net)
    
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

## 6. Transformer 组件

### 6.1 基础工具函数

In [ ]:
def masked_softmax(X, valid_lens):
    """带掩码的 softmax 操作"""
    if valid_lens is None:
        return F.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return F.softmax(X.reshape(shape), dim=-1)

### 6.2 注意力机制

In [ ]:
class DotProductAttention(nn.Module):
    """缩放点积注意力"""
    
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.attention_weights = None

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)


def transpose_qkv(X, num_heads):
    """为多头注意力并行计算而变换形状"""
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    """逆转 transpose_qkv 的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class MultiHeadAttention(nn.Module):
    """多头注意力模块（支持不同的 query_size, key_size, value_size）"""
    
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens=None):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)
        
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)
        
        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)

### 6.3 Transformer 核心组件

In [ ]:
class PositionalEncoding(nn.Module):
    """位置编码"""
    
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)


class PositionWiseFFN(nn.Module):
    """逐位置前馈网络"""
    
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs, **kwargs):
        super(PositionWiseFFN, self).__init__(**kwargs)
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))


class AddNorm(nn.Module):
    """残差连接 + 层归一化"""
    
    def __init__(self, normalized_shape, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

## 7. Transformer 编码器

In [ ]:
class Encoder(nn.Module):
    """编码器基类"""
    def __init__(self):
        super(Encoder, self).__init__()

    def forward(self, X, *args):
        raise NotImplementedError


class EncoderBlock(nn.Module):
    """Transformer 编码器块"""
    
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, use_bias=False, **kwargs):
        super(EncoderBlock, self).__init__(**kwargs)
        self.attention = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout, use_bias)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))


class TransformerEncoder(Encoder):
    """Transformer 编码器"""
    
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, use_bias=False, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block" + str(i),
                EncoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens,
                             num_heads, dropout, use_bias))

    def forward(self, X, valid_lens, *args):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X

## 8. Transformer 解码器

In [ ]:
class Decoder(nn.Module):
    """解码器基类"""
    def __init__(self):
        super(Decoder, self).__init__()

    def init_state(self, enc_outputs, *args):
        raise NotImplementedError

    def forward(self, X, state):
        raise NotImplementedError


class AttentionDecoder(Decoder):
    """带注意力的解码器基类"""
    def __init__(self):
        super(AttentionDecoder, self).__init__()

    @property
    def attention_weights(self):
        raise NotImplementedError


class DecoderBlock(nn.Module):
    """Transformer 解码器块"""
    
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, i, **kwargs):
        super(DecoderBlock, self).__init__(**kwargs)
        self.i = i
        self.attention1 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.attention2 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNorm(norm_shape, dropout)

    def forward(self, X, state):
        enc_outputs, enc_valid_lens = state[0], state[1]
        
        if state[2][self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[2][self.i], X), axis=1)
        state[2][self.i] = key_values
        
        if self.training:
            batch_size, num_steps, _ = X.shape
            dec_valid_lens = torch.arange(
                1, num_steps + 1, device=X.device).repeat(batch_size, 1)
        else:
            dec_valid_lens = None
        
        X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
        Y = self.addnorm1(X, X2)
        
        Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
        Z = self.addnorm2(Y, Y2)
        
        return self.addnorm3(Z, self.ffn(Z)), state


class TransformerDecoder(AttentionDecoder):
    """Transformer 解码器"""
    
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, **kwargs):
        super(TransformerDecoder, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block" + str(i),
                DecoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens,
                             num_heads, dropout, i))
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        return [enc_outputs, enc_valid_lens, [None] * self.num_layers]

    def forward(self, X, state):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self._attention_weights = [[None] * len(self.blks) for _ in range(2)]
        for i, blk in enumerate(self.blks):
            X, state = blk(X, state)
            self._attention_weights[0][i] = blk.attention1.attention.attention_weights
            self._attention_weights[1][i] = blk.attention2.attention.attention_weights
        return self.dense(X), state

    @property
    def attention_weights(self):
        return self._attention_weights

## 9. Encoder-Decoder 模型

In [ ]:
class EncoderDecoder(nn.Module):
    """编码器-解码器架构"""
    
    def __init__(self, encoder, decoder):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_all_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_all_outputs, *args)
        return self.decoder(dec_X, dec_state)[0]

## 10. 训练函数

In [ ]:
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """训练序列到序列模型"""
    
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)

    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    net.train()
    animator = Animator(xlabel='epoch', ylabel='loss', xlim=[10, num_epochs])
    
    for epoch in range(num_epochs):
        timer = Timer()
        metric = Accumulator(2)
        
        for batch in data_iter:
            optimizer.zero_grad()
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            
            bos = torch.tensor([tgt_vocab.bos] * Y.shape[0], device=device).reshape(-1, 1)
            dec_input = torch.cat([bos, Y[:, :-1]], 1)
            
            Y_hat = net(X, dec_input, X_valid_len)
            
            if isinstance(Y_hat, (tuple, list)):
                Y_hat = Y_hat[0]
            
            l = masked_softmax_ce_loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()
            grad_clipping(net, 1)
            num_tokens = Y_valid_len.sum()
            optimizer.step()
            
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
        
        if (epoch + 1) % 10 == 0:
            animator.add(epoch + 1, (metric[0] / metric[1],))
    
    print(f'loss {metric[0] / metric[1]:.3f}, {metric[1] / timer.stop():.1f} '
          f'tokens/sec on {str(device)}')
    animator.show()

## 11. 模型训练

In [ ]:
# ===== 超参数配置 =====
num_hiddens, num_layers, dropout, batch_size, num_steps = 32, 2, 0.1, 64, 10
lr, num_epochs, device = 0.005, 200, try_gpu()
ffn_num_input, ffn_num_hiddens, num_heads = 32, 64, 4
key_size, query_size, value_size = 32, 32, 32
norm_shape = [32]

# 加载英法翻译数据
train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)

# 构建 Transformer
encoder = TransformerEncoder(
    len(src_vocab), key_size, query_size, value_size, num_hiddens,
    norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
    num_layers, dropout)

decoder = TransformerDecoder(
    len(tgt_vocab), key_size, query_size, value_size, num_hiddens,
    norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
    num_layers, dropout)

net = EncoderDecoder(encoder, decoder)

# 开始训练
train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

## 12. 预测与 BLEU 评估

In [ ]:
def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab,
                    num_steps, device, save_attention_weights=False):
    """序列到序列预测（贪婪解码）"""
    net.eval()
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [src_vocab.eos]
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    src_tokens = truncate_pad(src_tokens, num_steps, src_vocab.pad)
    enc_X = torch.unsqueeze(torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
    
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    
    dec_X = torch.unsqueeze(torch.tensor([tgt_vocab.bos], dtype=torch.long, device=device), dim=0)
    output_seq, attention_weight_seq = [], []
    
    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)
        dec_X = Y.argmax(dim=2)
        pred = dec_X.squeeze(dim=0).type(torch.int32).item()
        
        if save_attention_weights:
            attention_weight_seq.append(net.decoder.attention_weights)
        
        if pred == tgt_vocab.eos:
            break
        output_seq.append(pred)
    
    return ' '.join(tgt_vocab.to_tokens(output_seq)), attention_weight_seq


def bleu(pred_seq, label_seq, k):
    """BLEU 评分计算"""
    pred_tokens, label_tokens = pred_seq.split(' '), label_seq.split(' ')
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    score = math.exp(min(0, 1 - len_label / len_pred))
    
    for n in range(1, min(k, len_pred) + 1):
        num_matches, label_subs = 0, collections.defaultdict(int)
        for i in range(len_label - n + 1):
            label_subs[' '.join(label_tokens[i: i + n])] += 1
        for i in range(len_pred - n + 1):
            if label_subs[' '.join(pred_tokens[i: i + n])] > 0:
                num_matches += 1
                label_subs[' '.join(pred_tokens[i: i + n])] -= 1
        score *= math.pow(num_matches / (len_pred - n + 1), math.pow(0.5, n))
    
    return score

In [ ]:
# 测试翻译
engs = ['go .', "i lost .", 'he\'s calm .', 'i\'m home .']
fras = ['va !', 'j\'ai perdu .', 'il est calme .', 'je suis chez moi .']

for eng, fra in zip(engs, fras):
    translation, dec_attention_weight_seq = predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device, True)
    print(f'{eng} => {translation}, bleu {bleu(translation, fra, k=2):.3f}')

## 13. 注意力权重可视化

In [ ]:
def show_heatmaps(matrices, xlabel, ylabel, titles=None, figsize=(7, 3.5), cmap='Reds'):
    """显示注意力热力图"""
    num_rows, num_cols = matrices.shape[0], matrices.shape[1]
    fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize,
                             sharex=True, sharey=True, squeeze=False)
    
    for i, (row_axes, row_matrices) in enumerate(zip(axes, matrices)):
        for j, (ax, matrix) in enumerate(zip(row_axes, row_matrices)):
            pcm = ax.imshow(matrix.cpu().numpy(), cmap=cmap)
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    
    fig.colorbar(pcm, ax=axes.ravel().tolist(), shrink=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# 编码器注意力热力图
enc_attention_weights = torch.cat(net.encoder.attention_weights, 0).reshape(
    (num_layers, num_heads, -1, num_steps))

show_heatmaps(
    enc_attention_weights.cpu(),
    xlabel='Key positions',
    ylabel='Query positions',
    titles=['Head %d' % i for i in range(1, 5)],
    figsize=(7, 3.5))

In [ ]:
# 解码器注意力权重
dec_attention_weights_2d = [head[0].tolist()
                            for step in dec_attention_weight_seq
                            for attn in step for blk in attn for head in blk]

dec_attention_weights_filled = torch.tensor(
    pd.DataFrame(dec_attention_weights_2d).fillna(0.0).values)

dec_attention_weights = dec_attention_weights_filled.reshape(
    (-1, 2, num_layers, num_heads, num_steps))

dec_self_attention_weights, dec_inter_attention_weights = \
    dec_attention_weights.permute(1, 2, 3, 0, 4)

# 解码器自注意力热力图
show_heatmaps(
    dec_self_attention_weights[:, :, :, :len(translation.split()) + 1],
    xlabel='Key positions', ylabel='Query positions',
    titles=['Head %d' % i for i in range(1, 5)], figsize=(7, 3.5))

In [ ]:
# 解码器交叉注意力热力图
show_heatmaps(
    dec_inter_attention_weights,
    xlabel='Key positions',
    ylabel='Query positions',
    titles=['Head %d' % i for i in range(1, 5)],
    figsize=(7, 3.5))